# Transformer Encoder — IMDB 감성 분류

## 이 노트북이 하는 일
IMDB 영화 리뷰(25000개 학습 + 25000개 테스트) 의 **긍정/부정 이진 분류**.

- 입력: 영화 리뷰 텍스트 (최대 600 토큰)
- 출력: `sigmoid` → 0 (부정) / 1 (긍정) 확률
- 모델: Embedding → Transformer Encoder → GlobalMaxPooling → Dropout → Dense(1, sigmoid)

## 두 개의 실험
1. **[A] Transformer without Positional Embedding** — 토큰 임베딩만 넣고 순서 정보 없이 학습
2. **[B] Transformer with Positional Embedding** — **학습형** Positional Embedding 추가

### 왜 이 비교가 중요한가
Attention 은 순서를 모름. 그런데 [A] 버전이 **꽤 잘 돌아가는** 이유는:
- GlobalMaxPooling 이 결국 토큰 집합 (bag-of-features) 로 요약
- IMDB 감성 분류는 **어순보다 어떤 단어가 있는지** 가 더 중요한 태스크
그래도 언어에서 순서는 무시 못 하므로 [B] 에서 Positional Embedding 을 추가해 성능 개선 폭을 확인.

### 학습형 vs 사인형 Positional Embedding
- 이 노트북은 `layers.Embedding` 을 써서 **학습 가능한 위치 벡터** 를 만드는 방식
- kinematics 노트북의 sin/cos 과는 다름
- 장점: 데이터에 맞게 최적화. 단점: `max_length` 까지만 작동하고 파라미터 수가 늘어남


---
# [A] Transformer Encoder (Positional Embedding 없음)

순서 정보 없이도 얼마나 가는지 확인하는 baseline.


## [A-1] 데이터 다운로드 & 검증 분할

### 셸 명령
- `!rm -r aclImdb` : 기존 폴더 정리 (중복 실행 대비)
- `!curl -O ...` : Stanford 서버에서 IMDB 원본 tar 다운로드
- `!tar -xf ...` : 압축 해제
- `!rm -r aclImdb/train/unsup` : 비지도학습용 리뷰 제거 (여기선 레이블된 25k만 사용)

### 왜 직접 디렉토리를 조작하는가
Keras 의 `text_dataset_from_directory` 는 **폴더 구조** 로 레이블을 추론한다:
```
aclImdb/
  train/
    pos/  ← 라벨 1 (긍정 리뷰 12500개)
    neg/  ← 라벨 0 (부정 리뷰 12500개)
  val/    ← 아래 셀에서 train 의 20% 를 이쪽으로 이동시킴
  test/
```
원본 데이터셋은 train/test 만 제공하므로, **train 의 20% 를 val 로 분리** 하는 로직을 직접 짠 것.

### 고정 시드 `random.Random(1337)` 의 이유
재현성. 매번 같은 파일 집합이 validation 으로 빠져야 실험 비교가 의미 있음.


In [ ]:
# [A-1] IMDB 데이터셋 준비 & val 분할

# 기존 디렉토리를 삭제 (재실행 대비)
!rm -r aclImdb

# IMDB 리뷰 데이터셋 다운로드 및 압축 해제
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

# 비지도학습 데이터셋(unsup)을 삭제 — 이 실험은 라벨된 데이터만 사용
!rm -r aclImdb/train/unsup

import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

batch_size = 32

# 데이터 디렉토리 경로 설정
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

# train → val 분할 (각 클래스마다 20% 를 val 로 이동)
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)                         # val 폴더 준비
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)                      # 고정 시드로 섞기 (재현성)
    num_val_samples = int(0.2 * len(files))                 # 20%
    val_files = files[-num_val_samples:]                    # 뒤쪽 20%
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)


## [A-2] tf.data 로 텍스트 로드 + 벡터화

### `text_dataset_from_directory`
폴더명이 자동으로 레이블. `(텍스트, 레이블)` 쌍의 `tf.data.Dataset` 을 반환.
`batch_size=32` 로 배치화.

### `TextVectorization` 레이어
- `max_tokens=20000` : 가장 자주 나오는 상위 20000 단어만 사용. 희귀 단어는 OOV 토큰(1) 로 치환
- `output_mode="int"` : 단어를 정수 인덱스로 변환 (embedding 에 들어가기 위해)
- `output_sequence_length=600` : 600 토큰으로 패딩/자르기 (고정 길이)
- `.adapt(text_only_train_ds)` : **train 데이터로만** vocabulary 학습 (val/test 누수 방지)

### 왜 모델 바깥에서 벡터화하고 `dataset.map` 으로 넣는가
- GPU 에서 학습하는 동안 CPU 가 전처리를 **병렬로** 수행 (`num_parallel_calls=4`) → 파이프라인 효율 ↑
- 동일한 vocabulary 를 train/val/test 에 일관 적용


In [ ]:
# [A-2] 텍스트 데이터셋 & 벡터화

# 디렉토리 구조 기반으로 dataset 생성 (라벨은 폴더명에서 자동 추론)
train_ds = keras.utils.text_dataset_from_directory("aclImdb/train", batch_size=batch_size)
val_ds   = keras.utils.text_dataset_from_directory("aclImdb/val",   batch_size=batch_size)
test_ds  = keras.utils.text_dataset_from_directory("aclImdb/test",  batch_size=batch_size)

# vocabulary 학습용으로 텍스트만 뽑은 dataset
text_only_train_ds = train_ds.map(lambda x, y: x)

# 텍스트 벡터화 설정
max_length = 600                                           # 긴 리뷰는 600 토큰으로 잘라냄
max_tokens = 20000                                         # 상위 20000 단어만 유지
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",                                     # 단어 → 정수 인덱스
    output_sequence_length=max_length,                     # 고정 길이 (패딩/자르기)
)
text_vectorization.adapt(text_only_train_ds)               # train 으로만 vocab 학습 (data leak 방지)

# 모든 split 에 동일한 벡터화 적용 — 병렬 map 으로 파이프라인 효율화
int_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds   = val_ds.map  (lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds  = test_ds.map (lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)


## [A-3] TransformerEncoder Layer (PE 없음)

### 왜 `layers.Layer` 서브클래스로 만드는가
`MultiHeadAttention + LayerNorm + FFN` 한 세트를 **재사용 가능한 모듈** 로 만들기 위함.
`get_config` 를 구현해서 모델 저장/로딩도 가능.

### 구조 (Post-Norm)
```
x ─ MHA ─ +x ─ LN1 ─ FFN ─ +(LN1 출력) ─ LN2 ─ out
```

### `build` 메서드
서브레이어들의 `build(input_shape)` 를 명시적으로 호출. Keras 3 의 strict build 요건 때문.
일반 함수형 블록에선 필요 없지만 커스텀 Layer 에서는 이렇게 해 줘야 weight 가 초기화됨.

### 마스크 처리
`text_dataset_from_directory` + `TextVectorization` 파이프라인에서는 패딩 토큰 0 이 들어옴.
`mask` 인자가 전달되면 `mask[:, tf.newaxis, :]` 로 shape 확장해서 attention 에 전달 → 패딩 토큰을 attention 계산에서 무시.


In [ ]:
# [A-3] Transformer Encoder 레이어 (PE 없음)
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads

        # 멀티헤드 셀프 어텐션
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        # FFN (2-layer MLP — 표현력 확장 → 복원)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1 = layers.LayerNormalization()     # attention 뒤
        self.layernorm_2 = layers.LayerNormalization()     # FFN 뒤

    def build(self, input_shape):
        # 서브레이어 수동 build (Keras 3 strict build 대응)
        self.attention.build(input_shape, input_shape)
        self.dense_proj.build(input_shape)
        self.layernorm_1.build(input_shape)
        self.layernorm_2.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, mask=None):
        # 패딩 토큰은 attention 에서 무시 (mask broadcast 형태로 변환)
        if mask is not None:
            mask = mask[:, tf.newaxis, :]                  # (B, T) → (B, 1, T)
        attention_output = self.attention(inputs, inputs, attention_mask=mask)

        proj_input  = self.layernorm_1(inputs + attention_output)   # residual + LN
        proj_output = self.dense_proj(proj_input)                   # FFN
        return self.layernorm_2(proj_input + proj_output)           # residual + LN

    def get_config(self):
        # 모델 저장 시 레이어 설정을 JSON 으로 저장하기 위함
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config


## [A-4] 모델 구성 + 학습

### 구조
```
단어 인덱스 (B, T)
  ↓ Embedding(vocab=20000, dim=256)
토큰 벡터 (B, T, 256)
  ↓ TransformerEncoder(256, 32, 2)
문맥화된 토큰 벡터 (B, T, 256)
  ↓ GlobalMaxPooling1D()            ← 시퀀스 중 가장 강한 신호만 뽑음
문서 벡터 (B, 256)
  ↓ Dropout(0.5)                    ← 과적합 완화
  ↓ Dense(1, sigmoid)               ← 이진 분류
```

### 하이퍼파라미터
- `embed_dim=256` : 단어 임베딩 차원. 256 은 IMDB 규모에 적당한 중간값
- `num_heads=2` : 작은 데이터라 헤드 많이 안 써도 됨
- `dense_dim=32` : FFN hidden. 작게 잡아서 과적합 방지 (IMDB 는 25k 학습 샘플이라 비교적 작음)
- `Dropout(0.5)` : 최종 분류 직전 강한 정규화

### 왜 GlobalMaxPooling 인가 (Average 가 아니라)
- MaxPooling 은 "가장 신호 강한 단어" 를 뽑아냄 → 감성 분류처럼 **특정 단어** ("awful", "amazing") 가 결정적인 태스크에 적합
- Averaging 은 모든 단어에 균등 가중 → 긴 리뷰의 신호가 희석

### ModelCheckpoint 로 최적 모델 저장
`save_best_only=True` → val_loss 가 개선될 때만 저장. 학습 후 **최고 성능 모델을 load** 해서 테스트.

### `binary_crossentropy` + `sigmoid` 조합
이진 분류의 표준. sigmoid 출력을 확률로 해석, BCE 로 학습.


In [ ]:
# [A-4] 모델 구성 & 학습
vocab_size = 20000
embed_dim  = 256
num_heads  = 2
dense_dim  = 32

inputs  = keras.Input(shape=(None,), dtype="int64")                        # 정수 토큰 시퀀스
x       = layers.Embedding(vocab_size, embed_dim)(inputs)                  # (B, T) → (B, T, 256)
x       = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)           # 문맥화
x       = layers.GlobalMaxPooling1D()(x)                                   # 가장 강한 신호 추출
x       = layers.Dropout(0.5)(x)                                           # 강한 정규화
outputs = layers.Dense(1, activation="sigmoid")(x)                         # 이진 분류

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# 학습 — 매 epoch 마다 val_loss 기준 최고 모델을 저장
callbacks = [
    keras.callbacks.ModelCheckpoint("transformer_encoder.keras", save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=20, callbacks=callbacks)

# 저장된 최고 모델을 다시 로드 — 커스텀 Layer 는 custom_objects 로 알려줘야 함
model = keras.models.load_model(
    "transformer_encoder.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder},
)
print(f"테스트 정확도: {model.evaluate(int_test_ds)[1]:.3f}")


---
# [B] Transformer Encoder + 학습형 Positional Embedding

[A] 에서 **순서 정보가 빠져있던 문제를 해결** 하는 버전.

### 학습형 Positional Embedding
`layers.Embedding(input_dim=max_length, output_dim=embed_dim)` 을 만들어
각 위치 (0, 1, ..., T-1) 마다 **학습 가능한 벡터** 를 할당.
입력 토큰 임베딩에 더해서 "이 단어가 몇 번째인지" 를 모델이 알게 함.

### 사인형(kinematics 노트북) vs 학습형 (이 노트북)
| | Sinusoidal PE | Learned PE |
|-|-|-|
| 파라미터 | 0개 | max_length × embed_dim |
| 유연성 | 고정 함수 | 데이터에 맞게 최적화 |
| 일반화 | 임의 길이 가능 | max_length 초과 불가 |
| 작은 데이터 | 유리 | 과적합 위험 |
| 큰 데이터 | 덜 fit | 성능 ↑ |

IMDB 는 25k 샘플로 그리 작지 않고 고정 길이 600 이라 학습형이 합리적인 선택.

### [A] 대비 실제 변화
- `TransformerEncoder.__init__` 에 `max_length` 인자 추가
- `self.position_embedding = layers.Embedding(max_length, embed_dim)` 생성
- `call()` 안에서 `positions = tf.range(seq_length)` → embedding → `inputs += embedded_positions` 로 PE 주입
- `build()` 에서 position_embedding 빌드, attention/FFN/LN 의 shape 을 명시적으로 `(B, T, embed_dim)` 으로 빌드


## [B-1] 데이터 재다운로드 & 재벡터화

[A] 실행 시 `val` 폴더를 만들어서 train 구조가 바뀌었으므로 다시 초기화.
나머지 로직은 [A-1], [A-2] 와 동일.


In [ ]:
# [B-1] 데이터 초기화 & 재로딩
!rm -r aclImdb
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

batch_size = 32

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)

train_ds = keras.utils.text_dataset_from_directory("aclImdb/train", batch_size=batch_size)
val_ds   = keras.utils.text_dataset_from_directory("aclImdb/val",   batch_size=batch_size)
test_ds  = keras.utils.text_dataset_from_directory("aclImdb/test",  batch_size=batch_size)
text_only_train_ds = train_ds.map(lambda x, y: x)

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds   = val_ds.map  (lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds  = test_ds.map (lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)


## [B-2] TransformerEncoder **with Positional Embedding**

### `position_embedding` 의 역할
```python
positions = tf.range(start=0, limit=seq_length, delta=1)   # [0, 1, 2, ..., T-1]
embedded_positions = self.position_embedding(positions)    # (T, embed_dim)
inputs += embedded_positions                               # broadcast 로 (B, T, embed_dim) 에 더함
```

**왜 더하기(+) 인가, concat 이 아니라?**
- 더하기는 차원을 늘리지 않음 → 이후 레이어의 입력 shape 유지
- 원 Transformer 논문의 관례 — "position 은 token 표현에 섞인다" 는 직관
- 모델이 학습하면서 임베딩 공간 내에서 "위치 부분공간" 과 "의미 부분공간" 을 분리해서 쓰게 됨


In [ ]:
# [B-2] Transformer Encoder with Positional Embedding
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, max_length=600, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.max_length = max_length

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

        # ★ 핵심 차이: 학습형 Positional Embedding
        #   각 위치 0~max_length-1 에 대해 embed_dim 차원 벡터를 학습
        self.position_embedding = layers.Embedding(
            input_dim=max_length, output_dim=embed_dim
        )

    def build(self, input_shape):
        seq_length = input_shape[1]
        self.position_embedding.build((seq_length,))

        # 명시적 build — Keras 3 strict build 대응
        self.attention.build(
            query_shape=(input_shape[0], seq_length, self.embed_dim),
            key_shape  =(input_shape[0], seq_length, self.embed_dim),
            value_shape=(input_shape[0], seq_length, self.embed_dim),
        )
        self.dense_proj.build((input_shape[0], seq_length, self.embed_dim))
        self.layernorm_1.build((input_shape[0], seq_length, self.embed_dim))
        self.layernorm_2.build((input_shape[0], seq_length, self.embed_dim))
        super().build(input_shape)

    def call(self, inputs, mask=None):
        # 입력: (B, T, embed_dim)
        seq_length = tf.shape(inputs)[1]

        # ★ 각 위치에 대응하는 학습형 임베딩 생성
        positions = tf.range(start=0, limit=seq_length, delta=1)          # (T,)
        embedded_positions = self.position_embedding(positions)           # (T, embed_dim)
        inputs += embedded_positions                                       # broadcast + 더하기

        # 이후는 [A-3] 과 동일
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(inputs, inputs, attention_mask=mask)
        proj_input  = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
            "max_length": self.max_length,
        })
        return config


## [B-3] 모델 구성 & 학습

[A-4] 와 거의 동일한 모델. 차이는 `TransformerEncoder` 가 내부적으로 PE 를 더해준다는 것뿐.

### 기대 결과
IMDB 에서 PE 유무 차이는 보통 **몇 % 포인트 accuracy** 정도.
태스크가 bag-of-features 에 강건해서 차이가 극적이진 않지만, 일반화 관점에서 PE 쪽이 보통 더 좋음.


In [ ]:
# [B-3] 모델 구성 & 학습
vocab_size = 20000
embed_dim  = 256
num_heads  = 2
dense_dim  = 32

inputs  = keras.Input(shape=(None,), dtype="int64")
x       = layers.Embedding(vocab_size, embed_dim)(inputs)
x       = TransformerEncoder(embed_dim, dense_dim, num_heads, max_length=max_length)(x)   # PE 포함된 encoder
x       = layers.GlobalMaxPooling1D()(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("transformer_encoder.keras", save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=20, callbacks=callbacks)

model = keras.models.load_model(
    "transformer_encoder.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder},
)
print(f"테스트 정확도: {model.evaluate(int_test_ds)[1]:.3f}")
